# Basics

Adapted from [ib_async basics](https://ib-api-reloaded.github.io/ib_async/notebooks.html).
Demonstrates ibx connection, account info, positions, and market data using the ibapi-compatible EClient/EWrapper pattern.

### Connecting

ibx connects directly to the brokerage gateway — no additional application needed.
Credentials are loaded from `.env` in the project root.

In [ ]:
import os, threading, time
from dotenv import load_dotenv
from ibx import EWrapper, EClient, Contract

load_dotenv()
USERNAME = os.environ["IB_USERNAME"]
PASSWORD = os.environ["IB_PASSWORD"]

In [ ]:
class App(EWrapper):
    def __init__(self):
        super().__init__()
        self.next_id = None
        self.account = None
        self.account_values = {}
        self.positions_list = []
        self.connected = threading.Event()

    def next_valid_id(self, order_id):
        self.next_id = order_id
        self.connected.set()

    def managed_accounts(self, accounts_list):
        self.account = accounts_list

    def account_summary(self, req_id, account, tag, value, currency):
        self.account_values[tag] = (value, currency)

    def account_summary_end(self, req_id):
        pass

    def position(self, account, contract, pos, avg_cost):
        self.positions_list.append((contract.symbol, contract.con_id, pos, avg_cost))

    def position_end(self):
        pass

    def error(self, req_id, error_code, error_string, advanced_order_reject_json=""):
        if error_code not in (2104, 2106, 2158):
            print(f"Error {error_code}: {error_string}")

In [ ]:
app = App()
client = EClient(app)
client.connect(username=USERNAME, password=PASSWORD, paper=True)

thread = threading.Thread(target=client.run, daemon=True)
thread.start()
app.connected.wait(timeout=10)

print(f"Connected: {client.is_connected()}")
print(f"Account: {app.account}")
print(f"Next order ID: {app.next_id}")

### Positions

Request current positions (delivered immediately from cached state):

In [ ]:
app.positions_list.clear()
client.req_positions()
time.sleep(0.5)

if app.positions_list:
    for symbol, con_id, pos, avg_cost in app.positions_list:
        print(f"  {symbol} (conId={con_id}): {pos:.0f} shares @ {avg_cost:.2f}")
else:
    print("No positions.")

### Account Values

Request account summary to get key values like Net Liquidation:

In [ ]:
client.req_account_summary(1, "All", "NetLiquidation,BuyingPower,TotalCashValue,UnrealizedPnL,RealizedPnL")
time.sleep(2)

for tag, (value, currency) in sorted(app.account_values.items()):
    print(f"  {tag}: {value} {currency}")

### Disconnecting

In [ ]:
client.disconnect()
print(f"Connected: {client.is_connected()}")